# Day 11: Ablation Studies — What Makes Memory Work?
**Goal:** Systematically test which design choices matter for the memory module's effectiveness. This is critical for the paper — reviewers will ask "why these hyperparameters?"

**Ablations we'll run:**
1. **Memory size**: 8 vs 16 vs 32 vs 64 vs 128 slots — how much past context is needed?
2. **Number of attention heads**: 1 vs 2 vs 4 vs 8 — how complex should retrieval be?
3. **Gate mechanism**: learned gate vs fixed gate vs no gate — does adaptive gating matter?
4. **Hidden dimension**: 256 vs 512 vs 768 — model capacity vs efficiency
5. **Sequence length**: 50 vs 100 vs 200 — training window length effect

Each ablation varies ONE factor while keeping everything else at the Day 10 defaults.

**Runtime:** A100 GPU (multiple training runs)

In [ ]:
# === INSTALL ===
!pip install torch torchvision torchaudio -q
!pip install nibabel nilearn scipy matplotlib seaborn -q
!pip install tqdm h5py scikit-learn huggingface_hub -q

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

import os, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import json, time, h5py, glob
from tqdm.auto import tqdm
from collections import OrderedDict

PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
FMRI_DIR = os.path.join(DATA_DIR, 'algonauts_fmri')
FEATURES_DIR = os.path.join(DATA_DIR, 'algonauts_features')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'day11_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Load Day 10 results for baseline
d10_path = os.path.join(PROJECT_DIR, 'day10_results', 'day10_results.json')
if os.path.exists(d10_path):
    with open(d10_path) as f:
        day10 = json.load(f)
    print(f"\nDay 10 baseline: ΔR = {day10['memory_benefit']['mean_delta_R']:+.4f}, "
          f"{day10['memory_benefit']['pct_improved']:.1f}% improved")

---
## 1. Load Data (same as Day 10)

In [ ]:
# Load fMRI
fmri_path = glob.glob(os.path.join(FMRI_DIR, '*friends*.h5'))[0]
with h5py.File(fmri_path, 'r') as f:
    for k in f.keys():
        if isinstance(f[k], h5py.Dataset) and len(f[k].shape) == 2:
            fmri_data = f[k][:]
            break
        elif isinstance(f[k], h5py.Group):
            for sk in f[k].keys():
                if isinstance(f[k][sk], h5py.Dataset) and len(f[k][sk].shape) == 2:
                    fmri_data = f[k][sk][:]
                    break

# Load features (try real first, then surrogate)
feature_files = (
    glob.glob(os.path.join(FEATURES_DIR, '**/*friends*.npy'), recursive=True) +
    glob.glob(os.path.join(FEATURES_DIR, '**/*friends*.npz'), recursive=True) +
    glob.glob(os.path.join(FEATURES_DIR, '**/*friends*.h5'), recursive=True) +
    glob.glob(os.path.join(FEATURES_DIR, '**/*sub-01*.npy'), recursive=True)
)

features = None
for ff in feature_files:
    try:
        if ff.endswith('.npy'):
            feat = np.load(ff)
        elif ff.endswith('.npz'):
            feat = np.load(ff)[list(np.load(ff).keys())[0]]
        elif ff.endswith('.h5'):
            with h5py.File(ff, 'r') as hf:
                for k in hf.keys():
                    if isinstance(hf[k], h5py.Dataset):
                        feat = hf[k][:]; break
        if len(feat.shape) == 2 and feat.shape[0] > 100:
            features = feat; break
    except:
        continue

if features is None:
    print("Using enhanced surrogate features...")
    from sklearn.decomposition import PCA, FastICA
    from scipy.ndimage import gaussian_filter1d
    from scipy.signal import hilbert
    np.random.seed(42)
    pca = PCA(n_components=50); pca_f = pca.fit_transform(fmri_data)
    ica = FastICA(n_components=30, random_state=42, max_iter=500); ica_f = ica.fit_transform(fmri_data)
    deriv_f = np.diff(pca_f, axis=0, prepend=pca_f[:1])
    slow_f = gaussian_filter1d(pca_f, sigma=10, axis=0)
    env_f = np.abs(hilbert(pca_f[:, :20], axis=0))
    combined = np.concatenate([pca_f, ica_f, deriv_f, slow_f, env_f], axis=1)
    proj = np.random.randn(combined.shape[1], 2048) * 0.05
    features = combined @ proj + gaussian_filter1d(np.random.randn(len(combined), 2048)*0.2, 1.5, axis=0)

# Align with HRF shift
hrf_shift = 4
min_len = min(len(features), len(fmri_data)) - hrf_shift
features = features[:min_len]
fmri_data = fmri_data[hrf_shift:hrf_shift+min_len]
features = (features - features.mean(0)) / (features.std(0) + 1e-8)
fmri_data = (fmri_data - fmri_data.mean(0)) / (fmri_data.std(0) + 1e-8)

n_trs = len(features)
feature_dim = features.shape[1]
n_parcels = fmri_data.shape[1]

# Split 80/20
nt = int(0.8 * n_trs)
train_f = torch.FloatTensor(features[:nt])
train_t = torch.FloatTensor(fmri_data[:nt])
val_f = torch.FloatTensor(features[nt:])
val_t = torch.FloatTensor(fmri_data[nt:])

print(f"Features: {features.shape}, fMRI: {fmri_data.shape}")
print(f"Train: {nt}, Val: {n_trs - nt}")

---
## 2. Model and Training Infrastructure

In [ ]:
class FeatureProjector(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class MemoryModule(nn.Module):
    def __init__(self, embed_dim=512, memory_size=64, n_heads=8, dropout=0.1, gate_type='learned'):
        super().__init__()
        self.memory_size = memory_size
        self.embed_dim = embed_dim
        self.gate_type = gate_type
        self.cross_attn = nn.MultiheadAttention(embed_dim, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim*4, embed_dim), nn.Dropout(dropout))
        if gate_type == 'learned':
            self.gate = nn.Parameter(torch.tensor(-2.0))
        elif gate_type == 'fixed_half':
            self.register_buffer('gate', torch.tensor(0.0))  # sigmoid(0)=0.5
        elif gate_type == 'fixed_one':
            self.register_buffer('gate', torch.tensor(5.0))  # sigmoid(5)≈1.0
        elif gate_type == 'none':
            self.register_buffer('gate', torch.tensor(5.0))
        self.memory_buffer = None
        self.memory_ptr = 0
        self.memory_count = 0

    def reset_memory(self, batch_size=1, device=None):
        if device is None: device = next(self.parameters()).device
        self.memory_buffer = torch.zeros(batch_size, self.memory_size, self.embed_dim, device=device)
        self.memory_ptr = 0; self.memory_count = 0

    def write_memory(self, x):
        buf = self.memory_buffer.clone()
        buf[:, self.memory_ptr] = x.detach()
        self.memory_buffer = buf
        self.memory_ptr = (self.memory_ptr + 1) % self.memory_size
        self.memory_count = min(self.memory_count + 1, self.memory_size)

    def forward(self, x):
        gate = torch.sigmoid(self.gate)
        if self.memory_buffer is None or self.memory_count == 0:
            self.write_memory(x); return x
        mem = self.memory_buffer[:, :self.memory_count]
        attn_out, _ = self.cross_attn(x.unsqueeze(1), mem, mem)
        attn_out = attn_out.squeeze(1)
        if self.gate_type == 'none':
            x_mem = self.norm1(x + attn_out)
        else:
            x_mem = self.norm1(x + gate * attn_out)
        x_mem = self.norm2(x_mem + self.ffn(x_mem))
        self.write_memory(x); return x_mem

class Encoder(nn.Module):
    def __init__(self, feat_dim, n_parcels=1000, hidden=512, mem_size=64,
                 n_heads=8, use_mem=True, dropout=0.1, gate_type='learned'):
        super().__init__()
        self.use_mem = use_mem
        self.projector = FeatureProjector(feat_dim, hidden, dropout)
        self.memory = MemoryModule(hidden, mem_size, n_heads, dropout, gate_type) if use_mem else None
        self.decoder = nn.Sequential(
            nn.Linear(hidden, hidden*2), nn.LayerNorm(hidden*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden*2, hidden*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden*2, n_parcels))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def reset_memory(self, bs=1):
        if self.memory: self.memory.reset_memory(bs, next(self.parameters()).device)

    def forward(self, x):
        e = self.projector(x)
        if self.memory: e = self.memory(e)
        return self.decoder(e)

def count_params(m): return sum(p.numel() for p in m.parameters())

def train_fast(model, tf, tt, vf, vt, n_epochs=20, lr=3e-4, seq_len=100):
    """Fast training for ablation — fewer epochs, returns key metrics."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_epochs)

    seqs_t = [(tf[s:s+seq_len], tt[s:s+seq_len]) for s in range(0, len(tf)-seq_len+1, seq_len//2)]
    seqs_v = [(vf[s:s+seq_len], vt[s:s+seq_len]) for s in range(0, len(vf)-seq_len+1, seq_len//2)]
    if not seqs_t: seqs_t = [(tf, tt)]
    if not seqs_v: seqs_v = [(vf, vt)]

    best_corr = -1
    for ep in range(n_epochs):
        model.train()
        for sf, st in seqs_t:
            sf, st = sf.to(device), st.to(device)
            model.reset_memory(1); opt.zero_grad()
            preds = torch.cat([model(sf[t:t+1]) for t in range(len(sf))], 0)
            F.mse_loss(preds, st).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()

        if (ep+1) % 5 == 0 or ep == n_epochs-1:
            model.eval()
            vp, vtg = [], []
            with torch.no_grad():
                for sf, st in seqs_v:
                    sf = sf.to(device); model.reset_memory(1)
                    vp.extend([model(sf[t:t+1]).cpu() for t in range(len(sf))])
                    vtg.append(st)
            vp = torch.cat(vp, 0).numpy(); vtg = torch.cat(vtg, 0).numpy()
            corrs = [np.corrcoef(vp[:,p], vtg[:,p])[0,1]
                     for p in range(vp.shape[1]) if vtg[:,p].std() > 1e-6]
            corrs = [c for c in corrs if not np.isnan(c)]
            mean_r = np.mean(corrs) if corrs else 0
            if mean_r > best_corr: best_corr = mean_r

    # Final eval
    model.eval()
    vp, vtg = [], []
    with torch.no_grad():
        for sf, st in seqs_v:
            sf = sf.to(device); model.reset_memory(1)
            vp.extend([model(sf[t:t+1]).cpu() for t in range(len(sf))])
            vtg.append(st)
    vp = torch.cat(vp, 0).numpy(); vtg = torch.cat(vtg, 0).numpy()
    corrs = np.array([np.corrcoef(vp[:,p], vtg[:,p])[0,1]
                      for p in range(vp.shape[1]) if vtg[:,p].std() > 1e-6])
    corrs = corrs[~np.isnan(corrs)]

    gate = torch.sigmoid(model.memory.gate).item() if model.memory else 0
    return {
        'mean_R': float(corrs.mean()),
        'median_R': float(np.median(corrs)),
        'top10_R': float(np.percentile(corrs, 90)),
        'bot10_R': float(np.percentile(corrs, 10)),
        'pct_positive': float((corrs > 0).mean() * 100),
        'gate': gate,
        'params': count_params(model),
        'parcel_corrs': corrs.tolist(),
    }

print("Model and training infrastructure ready!")

---
## 3. Run Ablation Studies

Each ablation trains a memory model AND a no-memory baseline, then computes the memory benefit (ΔR). We use 20 epochs instead of 30 to keep total runtime manageable (~30-40 min for all ablations).

In [ ]:
# First: train the no-memory baseline (shared across all ablations)
print("Training NO-MEMORY baseline...")
baseline_nomem = Encoder(feature_dim, n_parcels, 512, use_mem=False).to(device)
res_nomem = train_fast(baseline_nomem, train_f, train_t, val_f, val_t, n_epochs=20)
print(f"  Baseline R: {res_nomem['mean_R']:.4f}")
nomem_corrs = np.array(res_nomem['parcel_corrs'])

all_results = OrderedDict()
all_results['baseline_nomem'] = res_nomem

def run_ablation(name, **kwargs):
    """Train a memory model with given kwargs, compute delta vs baseline."""
    print(f"\n  [{name}]...", end=' ')
    model = Encoder(feature_dim, n_parcels, **kwargs).to(device)
    res = train_fast(model, train_f, train_t, val_f, val_t, n_epochs=20)
    mem_c = np.array(res['parcel_corrs'])
    nc = min(len(mem_c), len(nomem_corrs))
    delta = mem_c[:nc] - nomem_corrs[:nc]
    res['mean_delta_R'] = float(delta.mean())
    res['pct_improved'] = float((delta > 0).sum() / len(delta) * 100)
    res['config'] = name
    print(f"R={res['mean_R']:.4f}, ΔR={res['mean_delta_R']:+.4f}, "
          f"{res['pct_improved']:.0f}% improved, gate={res['gate']:.3f}")
    all_results[name] = res
    return res

print("\n" + "="*70)
print("ABLATION 1: Memory Size")
print("="*70)
for ms in [8, 16, 32, 64, 128]:
    run_ablation(f'mem_size_{ms}', hidden=512, mem_size=ms, n_heads=8, gate_type='learned')

print("\n" + "="*70)
print("ABLATION 2: Attention Heads")
print("="*70)
for nh in [1, 2, 4, 8]:
    run_ablation(f'heads_{nh}', hidden=512, mem_size=64, n_heads=nh, gate_type='learned')

print("\n" + "="*70)
print("ABLATION 3: Gate Mechanism")
print("="*70)
for gt in ['learned', 'fixed_half', 'fixed_one', 'none']:
    run_ablation(f'gate_{gt}', hidden=512, mem_size=64, n_heads=8, gate_type=gt)

print("\n" + "="*70)
print("ABLATION 4: Hidden Dimension")
print("="*70)
for hd in [256, 512, 768]:
    run_ablation(f'hidden_{hd}', hidden=hd, mem_size=64, n_heads=8, gate_type='learned')

print("\nAll ablations complete!")

In [ ]:
# ABLATION 5: Sequence Length (needs separate training)
print("="*70)
print("ABLATION 5: Sequence Length")
print("="*70)

for sl in [50, 100, 200]:
    name = f'seqlen_{sl}'
    print(f"\n  [{name}]...", end=' ')
    model = Encoder(feature_dim, n_parcels, hidden=512, mem_size=64, n_heads=8, gate_type='learned').to(device)
    res = train_fast(model, train_f, train_t, val_f, val_t, n_epochs=20, seq_len=sl)
    mem_c = np.array(res['parcel_corrs'])
    nc = min(len(mem_c), len(nomem_corrs))
    delta = mem_c[:nc] - nomem_corrs[:nc]
    res['mean_delta_R'] = float(delta.mean())
    res['pct_improved'] = float((delta > 0).sum() / len(delta) * 100)
    res['config'] = name
    print(f"R={res['mean_R']:.4f}, ΔR={res['mean_delta_R']:+.4f}, {res['pct_improved']:.0f}% improved")
    all_results[name] = res

print("\nAll ablations complete!")

---
## 4. Visualize Ablation Results

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(3, 2, hspace=0.4, wspace=0.3)

def plot_ablation(ax, prefix, xlabel, title):
    keys = [k for k in all_results if k.startswith(prefix)]
    if not keys: return
    labels = [k.replace(prefix+'_', '') for k in keys]
    delta_rs = [all_results[k]['mean_delta_R'] for k in keys]
    mean_rs = [all_results[k]['mean_R'] for k in keys]
    
    x = np.arange(len(labels))
    w = 0.35
    b1 = ax.bar(x - w/2, mean_rs, w, label='R (memory)', color='#2196F3', alpha=0.8)
    b2 = ax.bar(x + w/2, delta_rs, w, label='ΔR (benefit)', color='#4CAF50', alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_xlabel(xlabel); ax.set_ylabel('Correlation')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
    ax.axhline(0, color='k', lw=0.5)
    
    # Mark best
    best_idx = np.argmax(delta_rs)
    ax.bar(x[best_idx] + w/2, delta_rs[best_idx], w, color='#FF9800', alpha=0.9, edgecolor='black', lw=2)
    
    for i, (r, d) in enumerate(zip(mean_rs, delta_rs)):
        ax.text(i - w/2, r + 0.002, f'{r:.3f}', ha='center', fontsize=7)
        ax.text(i + w/2, max(d + 0.002, 0.002), f'{d:+.3f}', ha='center', fontsize=7)

# 1. Memory size
ax1 = fig.add_subplot(gs[0, 0])
plot_ablation(ax1, 'mem_size', 'Memory Slots', 'Ablation 1: Memory Size')

# 2. Attention heads
ax2 = fig.add_subplot(gs[0, 1])
plot_ablation(ax2, 'heads', 'Number of Heads', 'Ablation 2: Attention Heads')

# 3. Gate mechanism
ax3 = fig.add_subplot(gs[1, 0])
plot_ablation(ax3, 'gate', 'Gate Type', 'Ablation 3: Gate Mechanism')

# 4. Hidden dimension
ax4 = fig.add_subplot(gs[1, 1])
plot_ablation(ax4, 'hidden', 'Hidden Dim', 'Ablation 4: Hidden Dimension')

# 5. Sequence length
ax5 = fig.add_subplot(gs[2, 0])
plot_ablation(ax5, 'seqlen', 'Sequence Length (TRs)', 'Ablation 5: Sequence Length')

# 6. Summary table
ax6 = fig.add_subplot(gs[2, 1]); ax6.axis('off')

# Find best config for each ablation
summary_lines = ["BEST CONFIGURATIONS", "=" * 35, ""]
for prefix, label in [('mem_size', 'Memory Size'), ('heads', 'Attn Heads'),
                       ('gate', 'Gate Type'), ('hidden', 'Hidden Dim'), ('seqlen', 'Seq Length')]:
    keys = [k for k in all_results if k.startswith(prefix)]
    if keys:
        best_k = max(keys, key=lambda k: all_results[k]['mean_delta_R'])
        best = all_results[best_k]
        summary_lines.append(f"{label}:")
        summary_lines.append(f"  Best: {best_k.split('_',1)[1]}")
        summary_lines.append(f"  ΔR = {best['mean_delta_R']:+.4f}")
        summary_lines.append(f"  {best['pct_improved']:.0f}% improved")
        summary_lines.append("")

summary_lines.append(f"Baseline (no mem):")
summary_lines.append(f"  R = {res_nomem['mean_R']:.4f}")

ax6.text(0.05, 0.95, '\n'.join(summary_lines), transform=ax6.transAxes,
         fontsize=9, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.suptitle('Day 11: Ablation Studies — Memory Module Design Choices',
             fontsize=15, fontweight='bold', y=1.02)
plt.savefig(os.path.join(RESULTS_DIR, 'day11_ablation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Gate Dynamics Comparison

In [ ]:
# Compare gate values across ablations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gate values by memory size
ax = axes[0]
ms_keys = [k for k in all_results if k.startswith('mem_size')]
if ms_keys:
    sizes = [k.split('_')[-1] for k in ms_keys]
    gates = [all_results[k]['gate'] for k in ms_keys]
    ax.bar(sizes, gates, color='#4CAF50', alpha=0.8)
    ax.set_xlabel('Memory Size'); ax.set_ylabel('Final Gate Value')
    ax.set_title('Gate vs Memory Size')

# Gate values by heads
ax = axes[1]
h_keys = [k for k in all_results if k.startswith('heads')]
if h_keys:
    heads = [k.split('_')[-1] for k in h_keys]
    gates = [all_results[k]['gate'] for k in h_keys]
    ax.bar(heads, gates, color='#FF9800', alpha=0.8)
    ax.set_xlabel('Attention Heads'); ax.set_ylabel('Final Gate Value')
    ax.set_title('Gate vs Heads')

# Params vs performance
ax = axes[2]
all_mem_keys = [k for k in all_results if k != 'baseline_nomem']
if all_mem_keys:
    params = [all_results[k]['params']/1e6 for k in all_mem_keys]
    deltas = [all_results[k]['mean_delta_R'] for k in all_mem_keys]
    ax.scatter(params, deltas, c='#2196F3', alpha=0.7, s=60)
    for i, k in enumerate(all_mem_keys):
        ax.annotate(k.split('_',1)[1], (params[i], deltas[i]), fontsize=6, rotation=30)
    ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Mean ΔR')
    ax.set_title('Efficiency: Params vs Benefit')
    ax.axhline(0, color='r', ls='--', lw=0.5)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'day11_gate_and_efficiency.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Save Results

In [ ]:
# Compile all results
results = {
    'day': 11,
    'date': time.strftime('%Y-%m-%d'),
    'task': 'Ablation studies',
    'n_ablations': len(all_results) - 1,
    'baseline_nomem': {
        'mean_R': res_nomem['mean_R'],
        'params': res_nomem['params'],
    },
    'ablations': {},
    'best_configs': {},
}

for k, v in all_results.items():
    if k == 'baseline_nomem': continue
    results['ablations'][k] = {
        'mean_R': v['mean_R'],
        'mean_delta_R': v.get('mean_delta_R', 0),
        'pct_improved': v.get('pct_improved', 0),
        'gate': v['gate'],
        'params': v['params'],
    }

# Best per ablation category
for prefix, label in [('mem_size', 'memory_size'), ('heads', 'attention_heads'),
                       ('gate', 'gate_mechanism'), ('hidden', 'hidden_dim'), ('seqlen', 'sequence_length')]:
    keys = [k for k in all_results if k.startswith(prefix)]
    if keys:
        best_k = max(keys, key=lambda k: all_results[k]['mean_delta_R'])
        results['best_configs'][label] = {
            'config': best_k,
            'mean_delta_R': all_results[best_k]['mean_delta_R'],
            'mean_R': all_results[best_k]['mean_R'],
        }

results['next_steps'] = [
    'Day 12: Cross-subject generalization (Movie10 data)',
    'Day 13: Paper figures and results table',
    'Day 14: Paper draft outline and abstract',
]

with open(os.path.join(RESULTS_DIR, 'day11_results.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)

# Save detailed ablation data
with open(os.path.join(RESULTS_DIR, 'day11_all_ablations.json'), 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'parcel_corrs'}
               for k, v in all_results.items()}, f, indent=2, default=str)

print("Saved to:", RESULTS_DIR)
print("\n" + "="*60)
print("DAY 11 COMPLETE")
print("="*60)

print(f"\nRan {len(all_results)-1} ablation experiments")
print(f"\nBest configurations:")
for label, cfg in results['best_configs'].items():
    print(f"  {label}: {cfg['config']} (ΔR = {cfg['mean_delta_R']:+.4f})")

print(f"\nBaseline (no memory): R = {res_nomem['mean_R']:.4f}")
print(f"\nReady for Day 12: Cross-subject generalization!")